In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import sys
import ast
import json
import time
import logging
import warnings
import contextlib
import subprocess
import urllib.request
import concurrent.futures
from json import loads, dumps
from collections import defaultdict
from datetime import datetime, timedelta
from smtplib import SMTP
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pprint import pprint

import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import imgkit
from PIL import Image
from pretty_html_table import build_table
from requests_toolbelt.multipart.encoder import MultipartEncoder
from IPython.display import display, Markdown, Latex, HTML

from mars.analyzer.analysis import LighthouseAnalysis, GrafanaAnalysis
from mars.analyzer.algorithms import SNSigmaCI, NSigmaCI
from mars.analyzer.utils import draw_lighthouse_chart, create_lighthouse_query
from mars.datasource.lighthouse import LighthouseClient, LighthouseData, LighthouseQuery
from mars.datasource.sql2wrapper import query_sql2
from mars.datasource.timeseries import TimeSeries
from mars.datapreprocess import RemoveOutlier
from mars.datapreprocess.preprocess import lighthouse_value_to_df


# ============================================================
# Environment & display setup
# ============================================================
f = plt.figure()
f.set_figwidth(20)
f.set_figheight(5)
plt.rcParams['figure.figsize'] = [20, 5]
a = warnings.filterwarnings('ignore')
a = np.seterr(all="ignore")
devnull = open(os.devnull, 'w')
contextlib.redirect_stderr(devnull)
logging.basicConfig(level=logging.ERROR)
for handler in logging.root.handlers[:]:
    handler.setStream(devnull)
display(HTML("""
<style>
    .jp-CodeCell.jp-mod-outputsScrolled .jp-Cell-outputArea {max-height: 60em; }
</style>
"""))
cur_time = int(time.time())


# ============================================================
# Webex in-room orchestrator (5xx only: no Redis queue)
# ============================================================
# --- Webex in-room orchestrator (5xx only: no Redis queue) ---
# Slug must match Orchestrator agent name (see Delivery 5xx error analyzer.agent.md).
# Command shape: run <slug> -- <prompt>  (ASCII double hyphen, not em dash).
DELIVERY_5XX_AGENT_SLUG = os.environ.get(
    "COPILOT_5XX_ORCHESTRATOR_RUN", "delivery-5xx-error-analyzer-ereprod-akamcp-agent"
).strip()


def webex_log(message: str) -> None:
    print(f"[{datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}] {message}")


def orchestrator_mention_html() -> str:
    display = os.environ.get(
        "COPILOT_ORCHESTRATOR_DISPLAY_NAME", "Copilot-cli-orchestrator"
    ).strip()
    person_id = os.environ.get("COPILOT_ORCHESTRATOR_PERSON_ID", "").strip()
    if person_id:
        return f"<@personId:{person_id}|{display}>"
    email = os.environ.get(
        "COPILOT_ORCHESTRATOR_PERSON_EMAIL", "Copilot-cli-orchestrator@webex.bot"
    ).strip()
    return f"<@personEmail:{email}|{display}>"


def mention_command_separator() -> str:
    if "COPILOT_MENTION_COMMAND_SEPARATOR" in os.environ:
        return os.environ["COPILOT_MENTION_COMMAND_SEPARATOR"]
    return " "


def orchestrator_command_line(command: str) -> str:
    """Entire Webex body for Copilot orchestrator: mention + run … -- … only."""
    return f"{orchestrator_mention_html()}{mention_command_separator()}{command.strip()}"


def webex_plain_text_companion(markdown: str) -> str:
    out = markdown
    out = re.sub(r"<@personEmail:[^|>]+\|([^>]+)>", r"@\1", out)
    out = re.sub(r"<@personId:[^|>]+\|([^>]+)>", r"@\1", out)
    return out


def webex_send_room_markdown(token: str, room_id: str, markdown: str) -> dict:
    # NOTE: ca_verify is the Akamai-internal CA bundle and is NOT valid for
    # the public webexapis.com endpoint.  Use requests' default (certifi) CA.
    url = "https://webexapis.com/v1/messages"
    token_suffix = str(token)
    text_field = webex_plain_text_companion(markdown)
    payload = {"roomId": room_id, "markdown": markdown, "text": text_field}
    body = json.dumps(payload).encode("utf-8")
    webex_log(
        f"[webex_send_room_markdown] POST url={url} room_id={room_id} "
        f"token_suffix={token_suffix} body_bytes={len(body)}"
    )
    try:
        resp = requests.post(
            url,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
            },
            data=body,
            timeout=30,
        )
        if not resp.ok:
            webex_log(
                f"[webex_send_room_markdown] HTTPError code={resp.status_code} "
                f"room_id={room_id} token_suffix={token_suffix} body={resp.text}"
            )
            resp.raise_for_status()
        data = resp.json()
        webex_log(
            f"[webex_send_room_markdown] OK status={resp.status_code} "
            f"msg_id={data.get('id', '?')}"
        )
        return data
    except requests.exceptions.SSLError as e:
        webex_log(f"[webex_send_room_markdown] SSLError error={e} room_id={room_id} token_suffix={token_suffix}")
        raise
    except requests.exceptions.RequestException as e:
        webex_log(f"[webex_send_room_markdown] RequestException error={e} room_id={room_id} token_suffix={token_suffix}")
        raise


def build_5xx_orchestrator_command(orch_payload: dict) -> str:
    prompt = (
        f"Analyse the 5xx alert. product={orch_payload['product_name']} "
        f"service={orch_payload['service_name']} "
        f"data={json.dumps(orch_payload, separators=(',', ':'))}"
    )
    return f"run {DELIVERY_5XX_AGENT_SLUG} -- {prompt}"


def notify_5xx_orchestrator(token: str, room_id: str, orch_payload: dict) -> dict:
    """Post orchestrator-only line (separate from adaptive-card alert)."""
    return webex_send_room_markdown(
        token, room_id, orchestrator_command_line(build_5xx_orchestrator_command(orch_payload))
    )


# ============================================================
# Configuration & inputs
# ============================================================
# certificates that are required, change them accordingly
cert = ('/u0/abmukher/AGM/abmukher-20240417.crt', '/u0/abmukher/AGM/abmukher-20240417.key')
headers = {'Accept': 'application/json'}
ca_verify = '/u0/abmukher/AGM/ca-verify'
log = ""
cur_time = int(time.time())
print(cur_time)
product_data = ["amd","dd","ion","dsa","apiacc"]
service_data = ["W","S"]
namespace_input = "ereprod_health_review"
metric_data = ["global_f2b_x1000","global_f2h","global_edge_offload_pct","global_td0_offload_pct","global_origin_offload_pct",'global_edge_http_5xx_pct']
final_results_high = pd.DataFrame(columns=['metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','fail_severity'])
final_results_medium = pd.DataFrame(columns=['metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','fail_severity'])
final_results_low = pd.DataFrame(columns=['metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','fail_severity'])
metric_easy_name  = {
    "global_edge_hits":"Hits",
    "global_edge_bits": "Bits",
    "global_edge_flits":"Flits",
    "global_f2b_x1000" : "F2b",
    "global_f2h":"F2H",
    "global_edge_offload_pct":"Edge Offload",
    "global_td0_offload_pct" : "TD0 Offload",
    "global_origin_offload_pct" : "Origin Offload",
    "global_edge_http_5xx_pct": "5XX",
    "global_midgress_bits" : "Midgress Bits"
}


# --- helper: format a metric value with its unit ---
def format_metric_value(metric, value):
    if metric == "Hits":
        return str(round(value / (1000000),3)) + " " + "M"
    elif metric == "Bits" or metric == "Flits":
        return str(round(value / (1000**4),3)) + " " + "Tb/s"
    elif metric == "F2b":
        return str(round(value / (1000),3)) + " "
    elif metric == "F2H":
        return str(round(value / (1000),3)) + " " + "K"
    elif metric == "Midgress Bits":
        return str(round(value / (1000**3),3)) + " " + "Gb/s"
    else:
        return str(round(value,3)) + " %"


# ============================================================
# Metric analysis loop (collect & classify anomalies)
# ============================================================
for metric_input in metric_data:
    for product in product_data:
        for service in service_data:
            # --- skip product/service/metric combinations that are not applicable ---
            if product == "amd" and service == "S":
                continue
            if product == "dd" and service == "S":
                continue
            if "offload" in metric_input and product == "apiacc":
                continue
            if "offload" in metric_input and product == "ion":
                continue
            if product == "dsa" and service == "S" and "offload" in metric_input:
                continue
            log=log+f"Checking {metric_input},{product},{service}\n"

            # --- build the metric request for this combination ---
            analyse_d = [{'metric': f'{metric_input}', 'namespace': f'{namespace_input}', 'method': SNSigmaCI}]
            analyse_d = [analyse_d[0]]
            lighthouse_metrics = []
            namespace_l = []
            metrics_l = []
            for i in analyse_d:
                namespace_l.append(i['namespace'])
                metrics_l.append(i['metric'])
            lighthouse_metrics = [namespace_l] + [metrics_l]
            if "pct" not in metric_input and "midgress" not in metric_input:
                filter_json = {
                    "metric":  [
                        {
                            "name":'',
                            "tags": [
                                {
                                    "name": "product", "value": f"{product}"
                                },
                                {
                                    "name": "service", "value": f"{service}"
                                },
                                {
                                    "name": "waf", "value": "-"
                                },
                            ]
                        }
                    ]
                }
            else:
                filter_json = {
                    "metric":  [
                        {
                            "name":'',
                            "tags": [
                                {
                                    "name": "product", "value": f"{product}"
                                },
                                {
                                    "name": "service", "value": f"{service}"
                                },
                            ]
                        }
                    ]
                }

            # --- run the Lighthouse analysis ---
            #changed the data collection window
            data_col = 18
            trainwindow = 30
            # data_collect = data_col * 604800
            config = {
                'lighthouse_metrics': lighthouse_metrics,
                'cert': cert,
                'end_time': cur_time,
                # 'data_collection_window': 604800+data_collect,
                'default_method': {
                    'method': SNSigmaCI,
                    'train_end_time': cur_time - 604800,  # default: last datapoint in train set
                    'test_end_time': cur_time,  # default: last datapoint in train set
                    'period': 3600,
                    'test_window': 3600,
                    'train_window' : trainwindow*86400,
                    'clf': 3, # default: 3
                    'step' : 3600,
                    'blip_cycle': 1
                }
            }
            ga_client = LighthouseAnalysis(**config)
            ga_client.collect_data(j=32, filter=filter_json)
            result = ga_client.run_analysis()

            # --- convert epoch timestamps to readable strings ---
            def convert_timestamps_to_readable(timestamps):
                return [pd.to_datetime(ts, unit='s', utc=True).strftime('%Y-%m-%d %H:%M:%S') for ts in timestamps]

            result['failed_timestamp'] = result['failed_timestamp'].apply(convert_timestamps_to_readable)

            lh_result=result
            print(lh_result)

            # --- keep only failing datapoints with valid thresholds ---
            failed_results = lh_result[lh_result['result'] == 'FAIL'] #only fail result
            print(failed_results)
            failed_results = failed_results[failed_results['upper_threshold'].notnull()]
            failed_results = failed_results[failed_results['lower_threshold'].notnull()]
            print(failed_results.head())

            failed_results[~failed_results['metric'].str.contains('total_hits')].reset_index(drop=True)
            failed_results["lower_threshold"]  = failed_results["lower_threshold"].apply(lambda x: x[0])
            failed_results["upper_threshold"]  = failed_results["upper_threshold"].apply(lambda x: x[0])

            # --- drop datapoints whose severity is below threshold (< 3) ---
            timestamps = failed_results['failed_timestamp']
            sev = failed_results['fail_severity']
            count = 0
            l = []
            c=0
            if len(sev) == 0:
                continue
            for i in sev[0]:
                if i < 3:
                    l.append(count)
                    timestamps[0].pop(count)
                    c=c+1
                    continue
                c=c+1
                count += 1
            c=0
            for i in l:
                k=i-c
                sev[0].pop(i)
                c=c+1
            if len(timestamps[0])==0:
                continue
            failed_results['failed_timestamp'] = timestamps
            failed_results['fail_severity'] = sev
            filtered_list = []

            # --- normalise columns and map tags to "product,service" ---
            failed_results = failed_results[['metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','fail_severity']]
            failed_results['metric'] = failed_results['metric'].map(metric_easy_name).fillna(failed_results['metric'])
            for ll in range(0,len(failed_results)):
                json_roll = failed_results["tags"]
                json_roll = json_roll[0]
                semi_colon = []
                commas = []
                count = 0
                for colon in json_roll:
                    if colon == ":":
                        semi_colon.append(count)
                    if colon == ",":
                        commas.append(count)
                    count = count+1
                product_json = json_roll[semi_colon[0]+4:commas[0]-2]
                if len(commas)==2:
                    service_json = json_roll[semi_colon[1]+4:commas[1]-2]
                else:
                    service_json = json_roll[semi_colon[1]+4:len(json_roll)-3]
                string_json = product_json + "," + service_json
                failed_results["tags"] = string_json

            # --- format thresholds with units per metric ---
            metric_n = failed_results["metric"]
            lower_thres = float(failed_results["lower_threshold"][0])
            upper_thres = float(failed_results["upper_threshold"][0])
            lower_thres = [format_metric_value(metric_n[0], lower_thres)]
            upper_thres = [format_metric_value(metric_n[0], upper_thres)]

            # --- format failed datapoints with the same units ---
            temp_failed_data = failed_results["failed_datapoint"]
            temp_failed_datapoint = []
            counter_temp = 0
            for i in temp_failed_data[0]:
                temp_var = format_metric_value(metric_n[0], i)
                temp_failed_datapoint.append(temp_var)
                counter_temp += 1
            failed_results["failed_datapoint"] = [temp_failed_datapoint]
            failed_results["lower_threshold"] = lower_thres
            failed_results["upper_threshold"] = upper_thres

            # --- split failed datapoints into high / medium / low severity buckets ---
            failed_results_high = failed_results.copy()
            failed_results_medium = failed_results.copy()
            failed_results_low = failed_results.copy()
            time_stamp = failed_results_high["failed_timestamp"][0]
            fail_sev = failed_results_high["fail_severity"][0]
            fail_dat = failed_results_high["failed_datapoint"][0]
            index_high = []
            index_medium = []
            index_low = []
            for i in range(0,len(time_stamp)):
                if fail_sev[i]>5:
                    index_high.append(i)
                elif fail_sev[i]>4 and fail_sev[i]<=5:
                    index_medium.append(i)
                else:
                    index_low.append(i)
            if len(index_high) != 0:
                time_stamp_high = [value for lk, value in enumerate(time_stamp) if lk in index_high]
                fail_sev_high = [value for lk, value in enumerate(fail_sev) if lk in index_high]
                fail_dat_high = [value for lk, value in enumerate(fail_dat) if lk in index_high]
                failed_results_high["failed_timestamp"] = [time_stamp_high]
                failed_results_high["fail_severity"] = [fail_sev_high]
                failed_results_high["failed_datapoint"] = [fail_dat_high]
                high = pd.concat([final_results_high,failed_results_high],axis = 0)
                final_results_high = high
            if len(index_medium) != 0:
                time_stamp = failed_results_medium["failed_timestamp"][0]
                fail_sev = failed_results_medium["fail_severity"][0]
                fail_dat = failed_results_medium["failed_datapoint"][0]
                time_stamp_medium = [value for lk, value in enumerate(time_stamp) if lk in index_medium]
                fail_sev_medium = [value for lk, value in enumerate(fail_sev) if lk in index_medium]
                fail_dat_medium = [value for lk, value in enumerate(fail_dat) if lk in index_medium]
                failed_results_medium["failed_timestamp"] = [time_stamp_medium]
                failed_results_medium["fail_severity"] = [fail_sev_medium]
                failed_results_medium["failed_datapoint"] = [fail_dat_medium]
                medium = pd.concat([final_results_medium,failed_results_medium],axis = 0)
                final_results_medium = medium
            if len(index_low) != 0:
                time_stamp = failed_results_low["failed_timestamp"][0]
                fail_sev = failed_results_low["fail_severity"][0]
                fail_dat =  failed_results_low["failed_datapoint"][0]
                time_stamp_low = [value for lk, value in enumerate(time_stamp) if lk in index_low]
                fail_sev_low = [value for lk, value in enumerate(fail_sev) if lk in index_low]
                fail_dat_low = [value for lk, value in enumerate(fail_dat) if lk in index_low]
                failed_results_low["failed_timestamp"] = [time_stamp_low]
                failed_results_low["fail_severity"] = [fail_sev_low]
                failed_results_low["failed_datapoint"] = [fail_dat_low]
                low = pd.concat([final_results_low,failed_results_low],axis = 0)
                final_results_low = low


# ============================================================
# Build report & notify (only when anomalies were found)
# ============================================================
if final_results_high.empty and final_results_medium.empty and final_results_low.empty:
    log = "No anamoly \n \n"
else:
    # --------------------------------------------------------
    # Report builder helpers
    # --------------------------------------------------------
    # --- metric -> (time range, Grafana viewPanel id) ---
    GRAFANA_PANEL = {
        "Hits": ("now-2d", 62),
        "Bits": ("now-30d", 64),
        "Flits": ("now-30d", 97),
        "F2b": ("now-30d", 68),
        "F2H": ("now-30d", 70),
        "Edge Offload": ("now-30d", 56),
        "TD0 Offload": ("now-30d", 58),
        "Origin Offload": ("now-30d", 60),
        "Midgress Bits": ("now-30d", 66),
        "5XX": ("now-30d", 54),
    }
    GRAFANA_DEFAULT_URL = 'https://grafana-lh.akam.ai/d/oy86e2bIz/edge-networking-product-health-dashboard?orgId=1&from=now-7d&to=now&var-service=W&var-product=apiacc&var-product=amd&var-product=dd&var-product=ion&var-product=dsa'

    def grafana_panel_url(metric, service, product):
        if metric not in GRAFANA_PANEL:
            return GRAFANA_DEFAULT_URL
        time_range, view_panel = GRAFANA_PANEL[metric]
        return f'https://grafana-lh.akam.ai/d/oy86e2bIz/edge-networking-product-health-dashboard?orgId=1&from={time_range}&to=now&var-service={service}&var-product={product}&viewPanel={view_panel}'

    def generate_link(row):
        # Count the number of 'x's in the Condition column
        email_metric = row["metric"]
        email_tags = row["tags"]
        print(email_tags,"llll",row)
        email_ind_coma = email_tags.index(",")
        email_prod = email_tags[:email_ind_coma]
        email_service = email_tags[email_ind_coma+1:]
        print(email_prod,email_service,email_metric)
        # Determine the base URL based on the number of 'x's
        base_url = grafana_panel_url(email_metric, email_service, email_prod)
        print(base_url)
        # Create the hyperlink with the desired format
        link = f'<a href="{base_url}"> Link </a>'
        print(link)
        return link

    def process_row(row):
        if all(severity > 5 for severity in row['fail_severity']):
            row['fail_severity'] = 'high'
        elif all(4 <= severity <= 5 for severity in row['fail_severity']):
            row['fail_severity'] = 'medium'
        elif all(severity <= 4.5 for severity in row['fail_severity']):
            row['fail_severity'] = 'low'
        elif any(severity > 5 for severity in row['fail_severity']):
            row['fail_severity'] = 'high'
        elif any(severity > 3 for severity in row['fail_severity']):
            row['fail_severity'] = 'medium'
        elif any(severity > 0 for severity in row['fail_severity']):
            row['fail_severity'] = 'low'
        return row

    def breach(row):
        upper = row["upper_threshold"]
        low = row["lower_threshold"]
        spaceu = upper.index(" ")
        spacel = low.index(" ")
        upper = upper[:spaceu]
        low = low[:spacel]
        upper = float(upper)
        low = float(low)
        data = row["failed_datapoint"]
        thres = []
        for i in data:
            spaceb = i.index(" ")
            ex = i[:spaceb]
            ex = float(ex)
            if ex>upper:
                thres.append("Upper")
            if ex<low:
                thres.append("Lower")
        return thres

    def filter_and_split_dataframe(df):
        # Initialize the new dataframe
        filtered_data = {
            'metric': [],
            'tags': [],
            'failed_timestamp': [],
            'lower_threshold': [],
            'upper_threshold': [],
            'failed_datapoint': [],
            'fail_severity': [],
            'Breached_threshold': [],
        }

        # Process the dataframe
        for index, row in df.iterrows():
            metric = row['metric']
            breached_threshold = row['Breached_threshold']

            # Prepare new lists for updated rows
            new_failed_timestamp = []
            new_failed_datapoint = []
            new_fail_severity = []
            new_breached_threshold = []

            # Check for the condition and process the lists
            for i, threshold in enumerate(breached_threshold):
                if (metric == 'Flits' and threshold == 'Lower') or (metric == 'Bits' and threshold == 'Upper') or (metric == 'Hits' and threshold == 'Upper') or (metric == 'F2b' and threshold == 'Lower') or (metric == 'Edge Offload' and threshold == 'Upper') or (metric == 'TD0 Offload' and threshold == 'Upper') or (metric == 'Origin Offload' and threshold == 'Upper') or (metric == 'Midgress Bits' and threshold == 'Lower'):
                    # Add to the filtered dataframe
                    filtered_data['metric'].append(metric)
                    filtered_data['tags'].append(row['tags'])
                    filtered_data['failed_timestamp'].append(row['failed_timestamp'][i])
                    filtered_data['lower_threshold'].append(row['lower_threshold'])
                    filtered_data['upper_threshold'].append(row['upper_threshold'])
                    filtered_data['failed_datapoint'].append(row['failed_datapoint'][i])
                    filtered_data['fail_severity'].append(row['fail_severity'][i])
                    filtered_data['Breached_threshold'].append(threshold)
                else:
                    # Keep in the updated row
                    new_failed_timestamp.append(row['failed_timestamp'][i])
                    new_failed_datapoint.append(row['failed_datapoint'][i])
                    new_fail_severity.append(row['fail_severity'][i])
                    new_breached_threshold.append(threshold)

            # Update the original dataframe with the remaining elements
            df.at[index, 'failed_timestamp'] = new_failed_timestamp
            df.at[index, 'failed_datapoint'] = new_failed_datapoint
            df.at[index, 'fail_severity'] = new_fail_severity
            df.at[index, 'Breached_threshold'] = new_breached_threshold
        df = df[df['Breached_threshold'].map(len) > 0].reset_index(drop=True)
        # Create and return the filtered dataframe
        filtered_df = pd.DataFrame(filtered_data)
        return filtered_df

    def group_dataframe(df):
        # Group by the specified columns
        print(df)
        grouped_df = df.groupby(['metric', 'tags', 'upper_threshold', 'lower_threshold']).agg({
            'failed_timestamp': lambda x: list(x),
            'failed_datapoint': lambda x: list(x),
            'fail_severity': lambda x: list(x),
            'Breached_threshold': lambda x: list(x)
        }).reset_index()
        return grouped_df

    # --------------------------------------------------------
    # Email body header
    # --------------------------------------------------------
    current_time = datetime.now()
    time_24_hours_ago = current_time - timedelta(hours=24)
    current_time_str = current_time.strftime("%d/%m/%Y %H:%M:%S")
    time_24_hours_ago_str = time_24_hours_ago.strftime("%d/%m/%Y %H:%M:%S")
    body_content = "<br><br>Hi team,</br></br> <br> This is an automated alert from Grafana regarding an issue with the Product Health Dashboards. An anomaly has been detected in the metric, which may indicate a potential issue affecting performance. The alert provides the exact timestamp when the failure occurred with the upper and lower threshold that were calculated.  </br>"
    body_content = body_content+f'<p> The duration of the report is: {time_24_hours_ago_str} to {current_time_str}</p>'+'<p>There are 3 levels of severity: high, medium and low</p><p>Red is for Issues and Green for Improvements<p><table style="border: 1px solid black; border-collapse: collapse;"><tr><td style="border: 1px solid black; padding: 5px;background-color: #BF2600;">High</td><td style="border: 1px solid black; padding: 5px;background-color: #00875A;">More than 5 severity </td></tr><tr><td style="border: 1px solid black; padding: 5px;background-color: #FF7452;">Medium</td><td style="border: 1px solid black; padding: 5px;background-color: #57D9A3;">Between 4 to 5 severity </td></tr><td style="border: 1px solid black; padding: 5px;background-color: #FFBDAD;" bgcolor="#FFBDAD">Low</td><td style="border: 1px solid black; padding: 5px;background-color: #ABF5D1;">Less than or equal to 3 severity</td></table><p>follow the link to the documentation : <a href="https://collaborate.akamai.com/confluence/display/MSPFNDTN/Product+health+metrics+analysis+for+AGM+alerts#ProducthealthmetricsanalysisforAGMalerts-Metrics">Document</a></p>'
    log = "Anamoly is detected\n Sending mails\n"

    # --------------------------------------------------------
    # Prepare "bad" (degraded) metrics
    # --------------------------------------------------------
    bad_ones = pd.concat([final_results_high,final_results_medium,final_results_low])
    bad_ones = bad_ones.reset_index(drop=True)
    bad_ones = bad_ones[['metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','fail_severity']]
    bad_ones['Breached_threshold'] = bad_ones.apply(breach, axis=1)
    good_ones = filter_and_split_dataframe(bad_ones)
    bad_ones = bad_ones[bad_ones['Breached_threshold'].map(len) > 0].reset_index(drop=True)
    bad_ones = bad_ones.apply(process_row, axis=1)
    bad_ones['Grafana_Link'] = bad_ones.apply(generate_link, axis=1)

    def filter_failed_data(row):
        if row['metric'] == '5XX':
            # Convert failed_datapoint strings to numerical values
            failed_datapoint_values = [float(dp.strip(' %')) for dp in row['failed_datapoint']]
            # Identify indices to keep
            print([print(dp) for i, dp in enumerate(failed_datapoint_values)])
            indices_to_keep = [i for i, dp in enumerate(failed_datapoint_values) if dp >= 0.5]
            # Filter the lists based on the indices to keep
            row['failed_datapoint'] = [row['failed_datapoint'][i] for i in indices_to_keep]
            row['failed_timestamp'] = [row['failed_timestamp'][i] for i in indices_to_keep]
            row['Breached_threshold'] = [row['Breached_threshold'][i] for i in indices_to_keep]
        return row

    def remove_empty_failed_datapoint_rows(df):
        return df[df['failed_datapoint'].apply(lambda x: len(x) > 0)]

    bad_ones = bad_ones.apply(filter_failed_data, axis=1)
    bad_ones = remove_empty_failed_datapoint_rows(bad_ones)
    bad_ones_sent = bad_ones[['fail_severity','metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','Breached_threshold','Grafana_Link']]
    bad_ones_webex = bad_ones
    if bad_ones_webex.empty:
        x=1
        print("lolll")
    else:
        print(bad_ones_sent,"killlll")
        x=3

    # --------------------------------------------------------
    # Build "Issues" table (degraded metrics) for the email
    # --------------------------------------------------------
    bad_ones_web = build_table(bad_ones_webex, 'grey_dark' , odd_bg_color='white', font_size='14px',escape=False)
    #medium
    a = build_table(bad_ones_sent, 'grey_dark' , odd_bg_color='white', font_size='14px',escape=False)
    a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium',
              '<td style = "background-color: #FF7452!important;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium')
    a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium',
              '<td style = "background-color: #FF7452!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium')
    #low
    a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low',
             '<td style = "background-color: #FFBDAD!important;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low')
    a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low',
             '<td style = "background-color: #FFBDAD!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low')
    #high
    a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high',
             '<td style = "background-color: #BF2600!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high')
    a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high',
             '<td style = "background-color: #BF2600;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high')
    body_content = body_content+"<h3>Issues:</h3><p>The below table captures the metrics that have degraded in the last 24 hrs</p>"+a
    bad_content = a

    # --------------------------------------------------------
    # Build "Improvements" table (improved metrics) for the email
    # --------------------------------------------------------
    if good_ones.empty==False:
        good_ones = good_ones[good_ones['Breached_threshold'].map(len) > 0].reset_index(drop=True)
        good_ones = group_dataframe(good_ones)
        good_ones = good_ones.apply(process_row, axis=1)
        good_ones['Grafana_Link'] = good_ones.apply(generate_link, axis=1)
        good_ones_sent = good_ones[['fail_severity','metric','tags','failed_timestamp','lower_threshold','upper_threshold','failed_datapoint','Breached_threshold','Grafana_Link']]
        a = build_table(good_ones_sent, 'grey_dark', odd_bg_color='white', font_size='14px',escape=False)
        a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium',
                  '<td style = "background-color: #57D9A3!important;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium')
        a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium',
                  '<td style = "background-color: #57D9A3!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium')
        #low
        a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low',
                 '<td style = "background-color: #ABF5D1!important;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low')
        a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low',
                 '<td style = "background-color: #ABF5D1!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low')
        #high
        a = a.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high',
                 '<td style = "background-color: #00875A!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high')
        a = a.replace('<td style = "background-color: white;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high',
                 '<td style = "background-color: #00875A;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high')
        good_content = a
        body_content = body_content+"<h3>Improvements</h3><p>The below table captures the metrics that have improved in the last 24 hrs</p>"+a
    if good_ones.empty:
        body_content = body_content+"<h3>No Improvements were noticed</h3>"

    # --------------------------------------------------------
    # Compose & send the email
    # --------------------------------------------------------
    recipients = "dl-ereprod-info@akamai.com;hmuddapp@akamai.com;amuthann@akamai.com;abmukher@akamai.com"
    #recipients = "abmukher@akamai.com"
    sender ='dl-ereprod-oversight@akamai.com'

    message = MIMEMultipart()
    message['Subject'] = "Grafana Alert from Product health dashboard: Issue Detected at: " + \
              datetime.utcnow().strftime('%d, %b %Y %H:%M:%S UTC')
    message['From'] = sender
    message['To'] = recipients
    #medium
    body_content = body_content.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium',
                  '<td style = "background-color: #FF7452!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">medium')
    #low
    body_content = body_content.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low',
                 '<td style = "background-color: #FFBDAD!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">low')
    #high
    body_content = body_content.replace('<td style = "background-color: white; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high',
                 '<td style = "background-color: #BF2600!important; color: black;font-family: Century Gothic, sans-serif;font-size: 14px;text-align: left;padding: 0px 20px 0px 0px;width: auto">high')

    body_content = body_content.replace('<td style = "','<td style = "border: 1px solid black;')
    body_content = body_content+"<br>Thank you,</br>EREPROD Team"
    message.attach(MIMEText(body_content, "html"))
    if bad_ones.empty and good_ones.empty:
        sys.exit()
    msg_body = message.as_string()

    smtp_server = SMTP('relay.qa.akamai.com')
    smtp_server.sendmail(sender, recipients, msg_body)
    smtp_server.quit()
    print("Sending..",end="")
    for i in range(0,12):
        print("..",end="")
        time.sleep(1.2)
    print("Email has been sent")

    # --------------------------------------------------------
    # Webex notification (adaptive cards + 5XX orchestrator trigger)
    # --------------------------------------------------------
    def send_html_to_webex(text, webex_token, room_id, file, high_severity_df=pd.DataFrame()) :
        "Converts the html to image and sends to webex"
        webex_log(
            f"[send_html_to_webex] room_id={room_id} rows_in={len(high_severity_df)} "
            f"token_suffix={str(webex_token)[-10:]}"
        )
        print(room_id,"below is high")
        print(high_severity_df)
        if high_severity_df.empty==False:
            high_severity_df = high_severity_df[high_severity_df['fail_severity'].str.lower().isin(['high', 'medium'])]

            if high_severity_df.empty:
                return "No high severity failures detected."

            message_lines = ["🚨 **High Severity Failures**", "", " Please check the following issues:"]

            for idx, row in high_severity_df.iterrows():
                email_metric = row['metric']
                tag = row['tags'].split(",")
                print(tag,email_metric)
                base_url = grafana_panel_url(email_metric, tag[1], tag[0])

                # 5XX: orchestrator-only Webex post (mention + run … -- prompt). No Redis, no card.
                if row['metric'] == "5XX":
                    tags_split = row['tags'].split(',')
                    product_name = tags_split[0].strip()
                    service_name = tags_split[1].strip()
                    ts_list = row['failed_timestamp']
                    dp_list = row['failed_datapoint']
                    if isinstance(ts_list, str):
                        ts_list = eval(ts_list)
                    if isinstance(dp_list, str):
                        dp_list = eval(dp_list)
                    orch_payload = {
                        'time_ranges': {ts: dp for ts, dp in zip(ts_list, dp_list)},
                        'product_name': product_name,
                        'service_name': service_name,
                    }
                    try:
                        cmd = build_5xx_orchestrator_command(orch_payload)
                        webex_log(
                            f"[5xx-trigger] idx={idx} tags={row['tags']} room_id={room_id} cmd={cmd}"
                        )
                        resp = notify_5xx_orchestrator(webex_token, room_id, orch_payload)
                        webex_log(
                            f"✅ Orchestrator 5XX agent: {product_name}/{service_name} "
                            f"id={resp.get('id', '?')} room_id={room_id} orch_payload={orch_payload}"
                        )
                    except Exception as e:
                        webex_log(
                            f"❌ Orchestrator 5XX agent failed: {e} "
                            f"room_id={room_id} orch_payload={orch_payload}"
                        )

                text_block = (
                    f"-  ⌗ **Metric**: `{row['metric']}`\n"
                    f"-  ⌗ **Severity**: `{row['fail_severity']}`\n"
                    f"-  🏷️ **Tag**: `{row['tags']}`\n"
                    f"-  📅 **Time**: `{row['failed_timestamp']}`\n"
                    f"-  📉 **Datapoint**: `{row['failed_datapoint']}`\n"
                    f"-  ⚠️ **Threshold**: `{row['lower_threshold']} - {row['upper_threshold']}`"
                )

                # Webex API headers
                headers = {
                    'Authorization': f'Bearer {webex_token}',
                    'Content-Type': 'application/json'
                }

                # Adaptive card payload
                card_payload = {
                    "roomId": room_id,
                    "markdown": "🚨 Anomaly Alert!",
                    "attachments": [
                        {
                            "contentType": "application/vnd.microsoft.card.adaptive",
                            "content": {
                                "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
                                "type": "AdaptiveCard",
                                "version": "1.2",
                                "body": [
                                    {
                                        "type": "TextBlock",
                                        "text": text_block,
                                        "wrap": True,
                                        "size": "Medium",
                                        "weight": "Bolder"
                                    }
                                ],
                                "actions": [
                                    {
                                        "type": "Action.OpenUrl",
                                        "title": "Grafana",
                                        "url": base_url,
                                        "style": "positive"  # Renders as blue/green in Webex
                                    }
                                ]
                            }
                        }
                    ]
                }

                # Send card to Webex room
                response = requests.post(
                    url="https://webexapis.com/v1/messages",
                    headers=headers,
                    data=json.dumps(card_payload)
                )

                # Print result
                if response.status_code == 200:
                    webex_log("✅ Adaptive card sent successfully!")
                else:
                    webex_log(f"❌ Failed to send card: {response.status_code} body={response.text}")

    webex_token = "NDBlOWMzOWEtNDU4Yy00MDYwLThjMzMtMDZkM2M3ZWRhZDQxYjA4YTIwMDYtYTM0_PF84_a3749315-ae09-4a52-806c-2c3222fa7c2c"
    webex_room_id = "Y2lzY29zcGFyazovL3VzL1JPT00vZjRiNTA5OTAtMDBkMy0xMWYwLTgwZmEtMDkyYWU1ZTUzYTVj"
    #webex_room_id = "Y2lzY29zcGFyazovL3VzL1JPT00vNjU5YjgxMzAtMzNlNi0xMWYwLTg1MDUtMTExZGE1NmNkNWY3"

    def create_excel_file(df,pref):
        current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(current_datetime)
        file_path = f"{current_datetime}_{pref}_data.xlsx"
        df.to_excel(file_path, index=False)
        return file_path

    if bad_ones_sent.empty  == False:
        bad_file_path = create_excel_file(bad_ones_sent,"bad")
        current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        test = f"Bad state {current_datetime}\n"
        send_html_to_webex(test, webex_token, webex_room_id, bad_file_path, bad_ones_webex)

    log = log + "Subject of mail: Anamoly detected: " + datetime.utcnow().strftime('%d, %b %Y %H:%M:%S UTC') 